In [ ]:
import os
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter

from net import BERTClassifier
from data import LoadDataset
from train import train

In [ ]:
dataset_dir = "/root/autodl-tmp/BertClassifier/dataset"
dataset_path = "/root/autodl-tmp/BertClassifier/dataset/News_Category_Dataset_v3.json"
local_model_path = "/root/models/bert-base-uncased"

batch_size = 1024
num_workers = os.cpu_count() // 2
class_weights = True

dropout = 0.2

num_epochs = 5
lr={"bert": 1e-4, "outputs": 1e-3}

In [ ]:
train_set = LoadDataset(torch.load(f'{dataset_dir}/train.pt'))
test_set = LoadDataset(torch.load(f'{dataset_dir}/test.pt'))
c_weights = torch.load(f'{dataset_dir}/class_weights.pt')

# 加权重采样/降采样
train_labels = [data[-1].item() for data in train_set]
class_counts = Counter(train_labels)

sample_weights = [1.0 / class_counts[data[-1].item()] for data in train_set]
sampler = WeightedRandomSampler(sample_weights, num_samples=(len(sample_weights) * 2))

# train_iter = DataLoader(train_set, batch_size, shuffle=True, num_workers=num_workers)
train_iter = DataLoader(train_set, batch_size=batch_size, sampler=sampler)
test_iter = DataLoader(test_set, batch_size, shuffle=False, num_workers=num_workers)

In [ ]:
net = BERTClassifier(local_model_path, dropout)

# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha  # class weights
        self.gamma = gamma

    def forward(self, inputs, targets):
        self.alpha = self.alpha.to(inputs.device)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

loss = FocalLoss(c_weights, gamma=1.0)
# loss = nn.MSELoss()

params_1x = [param for name, param in net.named_parameters() if "bert" in name]
optimizer = torch.optim.Adam(
    [
        {'name': 'bert', 'params': params_1x, 'lr': lr["bert"]},
        {'name': 'outputs', 'params': net.output.parameters(), "lr": lr["outputs"]},
    ]
)

train(net, loss, optimizer, train_iter, test_iter, num_epochs, lr)

In [ ]:
import torch
from utils import f1_report
from net import BERTClassifier
from data import LoadDataset


local_model_path = "/root/models/bert-base-uncased"
param_path = "/root/autodl-tmp/BertClassifier/fine-tuning-bert.pth"
dataset_path = "/root/autodl-tmp/BertClassifier/dataset/News_Category_Dataset_v3.json"

dropout = 0.2
batch_size = 1024
device = "cuda:0"

net = BERTClassifier(local_model_path, dropout)
net.load_state_dict(torch.load(param_path, map_location=device, weights_only=True))
net = net.to(device)
net.eval()

test_set = LoadDataset(torch.load(f'{dataset_dir}/test.pt'))
test_iter = DataLoader(test_set, batch_size, shuffle=False, num_workers=num_workers)

f1_report(net, test_iter)

In [ ]:
import torch
from net import BERTClassifier

dropout = 0.2
device = "cpu"
local_model_path = "/root/models/bert-base-uncased"
param_path = "/root/autodl-tmp/BertClassifier/fine-tuning-bert.pth"


net = BERTClassifier(local_model_path, dropout)
net.load_state_dict(torch.load(param_path, map_location=device, weights_only=True))
net.eval()

tokens_X = torch.randint(0, 1000, (1, 40))  # tokens_X
segments_X = torch.randint(0, 2, (1, 40))  # segments_X
valid_lens_X = torch.ones((1, 40), dtype=torch.long)  # valid_lens_x
valid_lens_X[:, -10:] = 0
dummy_input = (tokens_X, segments_X, valid_lens_X)

torch.onnx.export(
    net,
    dummy_input,
    "fine-tuning-bert.onnx",
    input_names=['tokens_X', 'segments_X', 'valid_lens_X'],
    output_names=["output"],
    dynamic_shapes={
        'tokens_X': {0: 'batch_size', 1: 'sequence_length'},
        'segments_X': {0: 'batch_size', 1: 'sequence_length'},
        'valid_lens_X': {0: 'batch_size', 1: 'sequence_length'}
    },
    opset_version=18,
    do_constant_folding=True,  # 优化常量折叠
    verbose=True  # 打印导出日志
)

print("导出成功！")

In [ ]:
import torch
import onnxruntime as ort
import numpy as np
from transformers import BertTokenizer

# 加载ONNX模型
ort_session = ort.InferenceSession("fine-tuning-bert.onnx")
tokenizer = BertTokenizer.from_pretrained("/root/models/bert-base-uncased")

# 准备真实推理数据（使用tokenizer）
text = "This movie is great!"
encoded = tokenizer(text, truncation=True, padding='max_length', max_length=40)

# 转换为numpy
inputs = {
    'tokens_X': np.array(encoded['input_ids'], dtype=np.int64).reshape(1, -1),
    'segments_X': np.array(encoded['token_type_ids'], dtype=np.int64).reshape(1, -1),
    'valid_lens_X': np.array(encoded['attention_mask'], dtype=np.int64).reshape(1, -1)
}

# ONNX推理
onnx_output = ort_session.run(['output'], inputs)[0]

# 与PyTorch模型对比
with torch.no_grad():
    pt_output = net(
        torch.tensor(encoded['input_ids'], device="cuda:0").reshape(1, -1),
        torch.tensor(encoded['token_type_ids'], device="cuda:0").reshape(1, -1),
        torch.tensor(encoded['attention_mask'], device="cuda:0").reshape(1, -1)
    ).cpu().numpy()

# 验证一致性
print(f"PyTorch输出: {pt_output}")
print(f"ONNX输出: {onnx_output}")
print(f"最大差异: {np.abs(pt_output - onnx_output).max()}")  # 应该接近0

In [ ]:
import onnxruntime as ort

ort_session = ort.InferenceSession("fine-tuning-bert.onnx")

In [ ]:
import onnxruntime as ort
from transformers import BertTokenizer

ort_session = ort.InferenceSession("fine-tuning-bert.onnx")
tokenizer = BertTokenizer.from_pretrained("/root/models/bert-base-uncased")

texts = [
    "This movie is great!",
    "I really enjoyed this film",
    "Not my favorite, but okay"
]
encoded = tokenizer(
    texts, 
    truncation=True, 
    padding='max_length', 
    max_length=40,
    return_tensors='np'  # 直接返回numpy数组
)

inputs = {
    'tokens_X': encoded['input_ids'],
    'segments_X': encoded['token_type_ids'],
    'valid_lens_X': encoded['attention_mask']
}

onnx_output = ort_session.run(['output'], inputs)[0]
onnx_output